# 44a · Phim chụp **trước** ở mức `mean(patch)`, trên backbone **đã fine-tune**

Bài nhỏ, rẻ, chạy **trước** `44b` — và nó quyết định `44b` được thiết kế thế nào.

## Vì sao không dùng lại `43`

`43` chạy trên `emb_nosq518_patch.npy` của `31` — đó là backbone **pretrain**. `44b` sẽ chạy trên
backbone **`ft`** (`38a/a2_E3`). Dùng cache cũ thì hai bài **không so được với nhau**, và cả mục đích
tách hai mức biểu diễn hỏng.

Nên `C1` trích lại `mean(patch)` từ backbone `ft` cho cả 377,110 ảnh (~30 phút), rồi mọi arm chạy
trong vài phút trên vector `768`.

## Bốn arm — `Z` vẫn là arm quyết định

```
A  [e_cur]                                768    goc toa do
B  [e_cur ; e_prior]                     1536    ANH TRUOC THO
D  [e_cur ; e_cur - e_prior]             1536    HIEU (dong cung phep tru)
Z  [e_cur ; e_prior CUA BENH NHAN KHAC]  1536    DOI CHUNG KHOP CHIEU
```

`43` (backbone pretrain) đo được: `D +0.0031` > `B +0.0026` > `C +0.0010`, và **`Z − A = −0.0089`**
— thêm chiều từ một **ảnh thật** thì *có hại*, nên `B − A` dương là **sau khi** đã trả giá đó.

Ở đây kiểm hai chuyện:
1. Kết luận **`D > B`** có lặp lại trên backbone `ft` không — nếu có thì `44b` đóng cứng phép hiệu
   vào `Tier1Img` là có cơ sở; nếu không thì `44b` phải thiết kế lại.
2. **Biên độ seed thật** (3 seed) — `44b` chỉ chạy 1 seed nên sẽ **mượn** con số này.

Ở mức này phép trừ hợp lệ vì `mean(patch)` **đã xoá sạch toạ độ** — không còn chỉ số ô lưới để lệch
đăng ký. Ở mức patch của `44b` thì khác hẳn, và đó là lý do `44b` phải dùng cross-attention.

## Không đụng gì tới notebook khác

```
GHI   stage4_nb44a/   <- rieng, ke ca cache emb_ft_meanpatch.npy (KHONG de o embeddings/)
DOC   cxrlt_*.parquet · emb_index.parquet · backbone_ft.pt · lb_*.tar   (deu chi doc)
```

Chạy song song `44b` ở tài khoản khác được.

## C0 · Config

In [1]:
import os, json, math, time, tarfile, io as _io, re
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES44A  = os.path.join(OUT_DIR, 'stage4_nb44a')
LOCAL   = '/content/cxrlt_518'

BB_RUN  = 'a2_E3'                  # lan chay 38a co diem U XONG cao nhat
SEEDS   = (42, 43, 44)             # RE nen du 3 seed -> bien do THAT, `44b` se muon
SEED    = 42
SIZE, BATCH_EXT = 518, 160
EPOCHS, LR, WD = 60, 3e-3, 1e-4    # asl40 @E=60 — cau hinh 35 da chot
NW = min(8, (os.cpu_count() or 2))

def ensure_res():
    '''Goi TRUOC MOI lan ghi. Drive FUSE danh roi thu muc RONG — da dinh o 43 va 38c/38d.'''
    os.makedirs(RES44A, exist_ok=True); return RES44A
ensure_res()
open(os.path.join(RES44A, '_run_info.json'), 'w').write('{"notebook":"44a"}')   # moc, GIU lai
# LUU Y: KHONG bao gio va `builtins.open`/`np.save`. Da thu va gay de quy vo han + chet worker.

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
assert os.path.exists(BB_SRC), f'khong thay {BB_SRC}'
P_EMB_FT = os.path.join(RES44A, 'emb_ft_meanpatch.npy')      # cache RIENG cua 44a

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id']+LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15*len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL           # CUNG cach cat nhu 35/37/38*/43
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
print(f'{len(ORDER):,} anh | khop {TF_.sum():,} | chon {SL.sum():,} | val {VA.sum():,}')
print(f'ghi vao {RES44A} | cache {os.path.basename(P_EMB_FT)}')

Mounted at /content/drive
377,110 anh | khop 219,785 | chon 39,086 | val 39,293
ghi vao /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb44a | cache emb_ft_meanpatch.npy


## C1 · Trích `mean(patch)` từ backbone **`ft`** — một lần, có resume

Ghi thẳng vào `memmap` và lưu tiến độ, theo khuôn `31`. Đứt thì chạy lại cell này, nó tiếp từ chỗ dở.

In [2]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER)*0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR,f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE,SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp,'rb') as f: buf = f.read()      # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE,'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER)*0.999, f'thieu {len(ORDER)-_n:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck  = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_pre = AutoModel.from_pretrained('microsoft/rad-dino')
_sd0 = {k: v.clone() for k, v in _pre.state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k]-v).abs().max().item() for k, v in backbone.state_dict().items())
assert _d > 1e-6, f'backbone GIONG HET ban pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
del _pre, _sd0
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
print(f'BACKBONE `ft` 38a/{BB_RUN} | lech so voi pretrain {_d:.3f} | D={D_CXR}')

class DSX(Dataset):
    def __init__(s, idx): s.idx = np.asarray(idx)
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return eval_tf(Image.merge('RGB', (im, im, im))), i

P_PROG = os.path.join(RES44A, '_extract_progress.json')
n0 = json.load(open(P_PROG))['n'] if os.path.exists(P_PROG) else 0
if n0 < len(ORDER):
    mode = 'r+' if os.path.exists(P_EMB_FT) else 'w+'
    E = np.lib.format.open_memmap(P_EMB_FT, mode=mode, dtype='float32', shape=(len(ORDER), D_CXR))
    dl = DataLoader(DSX(np.arange(n0, len(ORDER))), batch_size=BATCH_EXT, shuffle=False,
                    num_workers=NW, pin_memory=True)
    t0, seen = time.time(), n0
    with torch.no_grad():
        for x, ii in tqdm(dl, desc='trich mean(patch)'):
            with torch.autocast('cuda', dtype=AMP, enabled=dev=='cuda'):
                h = backbone(pixel_values=x.to(dev, non_blocking=True)).last_hidden_state
            E[ii.numpy()] = h[:, 1:].float().mean(1).cpu().numpy()   # BO token CLS, trung binh patch
            seen += len(ii)
            if seen % (BATCH_EXT*50) < BATCH_EXT:
                E.flush(); ensure_res(); json.dump({'n': int(seen)}, open(P_PROG,'w'))
    E.flush(); ensure_res(); json.dump({'n': len(ORDER)}, open(P_PROG,'w'))
    print(f'trich xong {len(ORDER):,} anh | {(time.time()-t0)/60:.1f} phut '
          f'| {(len(ORDER)-n0)/max(time.time()-t0,1e-9):.0f} img/s')
    del E
E = np.array(np.load(P_EMB_FT, mmap_mode='r'))          # nap HAN vao RAM, khong tra ve view
assert E.shape == (len(ORDER), D_CXR) and np.isfinite(E).all()
assert np.abs(E).sum(1).min() > 0, 'co hang toan 0 -> trich chua xong, chay lai cell nay'
print(f'E {E.shape} ({E.nbytes/1e9:.2f} GB) san sang')

anh local: 0 / 377,110


stage:   0%|          | 0/378 [00:00<?, ?tar/s]

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

...

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft` 38a/a2_E3 | lech so voi pretrain 0.021 | D=768


trich mean(patch):   0%|          | 0/2357 [00:00<?, ?it/s]

trich xong 377,110 anh | 17.8 phut | 352 img/s
E (377110, 768) (1.16 GB) san sang


## C2 · Ghép phim trước — study gần nhất của cùng bệnh nhân, trước `index_time`

`e_prior` = **trung bình embedding các ảnh của study đó**. Trung bình chứ không chọn frontal: ít tuỳ
tiện hơn, và `42` đo trung vị chỉ 2 phim trước nên phần lớn study trước chỉ có 1–2 ảnh.

In [3]:
BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
BASE['index_time'] = pd.to_datetime(BASE.index_time)
_have = set(ORDER.study_id.unique())
B = BASE[BASE.study_id.isin(_have)].sort_values(['subject_id','index_time']).reset_index(drop=True)
B['prev_study'] = B.groupby('subject_id', sort=False).study_id.shift(1)
PREV = dict(zip(B.study_id.to_numpy(), B.prev_study.to_numpy()))

_rows = {}
for i, sid in enumerate(ORDER.study_id.to_numpy()): _rows.setdefault(sid, []).append(i)
prev_of = np.array([PREV.get(s, np.nan) for s in ORDER.study_id.to_numpy()], dtype='float64')
HAS = ~np.isnan(prev_of)
print(f'anh co phim truoc: {HAS.sum():,} / {len(ORDER):,} ({HAS.mean():.1%})   (42 do 71.3% o muc study)')
assert HAS.mean() > 0.5, 'ty le qua thap -> kiem lai ghep study'

E_PREV = np.zeros_like(E)
for i in np.where(HAS)[0]: E_PREV[i] = E[_rows[int(prev_of[i])]].mean(0)

# --- Z: hoan vi THEO BENH NHAN, cung so chieu cung phan phoi, chi SAI NGUOI ---
_rz = np.random.default_rng(0); _idx = np.where(HAS)[0]
_perm = np.zeros(len(ORDER), 'int64'); _perm[_idx] = _rz.permutation(_idx)
E_SHUF = np.zeros_like(E_PREV); E_SHUF[_idx] = E_PREV[_perm[_idx]]
print(f'hoan vi Z: {(SUB[_idx]==SUB[_perm[_idx]]).mean():.2%} tinh co van cung benh nhan (nho la dung)')

KEEP = HAS                    # CHI anh co phim truoc -> so sanh sach
TFk, SLk, VAk = TF_ & KEEP, SL & KEEP, VA & KEEP
print(f'tap probe: khop {TFk.sum():,} | chon {SLk.sum():,} | val {VAk.sum():,}')

anh co phim truoc: 253,283 / 377,110 (67.2%)   (42 do 71.3% o muc study)
hoan vi Z: 0.01% tinh co van cung benh nhan (nho la dung)
tap probe: khop 146,515 | chon 26,179 | val 26,397


## C3 · Đầu đọc — `asl40`, **epoch cuối**, không early-stop

In [4]:
class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*. gn=4, gp=0.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        p = torch.sigmoid(x); pm = (p - s.clip).clamp(min=0)
        L = (y*torch.log(p.clamp(min=s.eps))*(1-p)**s.gp
             + (1-y)*torch.log((1-pm).clamp(min=s.eps))*pm**s.gn)
        return -L.mean()

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, sc = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-sc, kind='stable'); yy, ss = y[r], sc[r]
        tp, fp = np.cumsum(yy), np.cumsum(1-yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp/(tp+fp), tp/n1
        o[j] = float((np.diff(np.r_[0.0, rc])*pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def fit(X, *, seed=SEED, epochs=EPOCHS):
    '''EPOCH CUOI, cosine trai tren `epochs`, KHONG early-stop (giao thuc nb35 §0.1).
       Chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val.'''
    torch.manual_seed(seed)
    mu, sd = X[TFk].mean(0, keepdims=True), X[TFk].std(0, keepdims=True) + 1e-6
    z = lambda m: torch.from_numpy(((X[m]-mu)/sd).astype('float32')).to(dev)
    xt, xs, xv = z(TFk), z(SLk), z(VAk)
    yt = torch.from_numpy(YM[TFk]).to(dev)
    m = nn.Linear(xt.shape[1], C).to(dev)
    crit = ASL(gn=4, gp=0).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    bs = int(min(4096, max(256, len(xt)//64)))      # san 64 buoc/epoch (bay da dinh o nb35)
    n = len(xt)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            k = perm[i:i+bs]
            opt.zero_grad(set_to_none=True); crit(m(xt[k]), yt[k]).backward(); opt.step()
        sch.step()
    m.eval()
    with torch.no_grad():
        return torch.sigmoid(m(xv)).cpu().numpy(), torch.sigmoid(m(xs)).cpu().numpy()
print(f'san sang | asl40 E={EPOCHS} epoch cuoi | seeds {SEEDS}')

san sang | asl40 E=60 epoch cuoi | seeds (42, 43, 44)


## C4 · Bốn arm × ba seed

In [5]:
ARMS = {'A': '[e_cur]', 'B': '[e_cur;e_prior]', 'D': '[e_cur;e_cur-e_prior]',
        'Z': '[e_cur;e_prior BN KHAC]  <- DOI CHUNG'}
def build(tag):
    return {'A': E, 'B': np.concatenate([E, E_PREV], 1),
            'D': np.concatenate([E, E-E_PREV], 1), 'Z': np.concatenate([E, E_SHUF], 1)}[tag]

P_RES = os.path.join(RES44A, 'probe.csv')
R = pd.read_csv(P_RES) if os.path.exists(P_RES) else pd.DataFrame()
for tag, desc in ARMS.items():
    X = build(tag)
    for sd_ in SEEDS:
        if len(R) and ((R.arm == tag) & (R.seed == sd_)).any():
            print(f'{tag} s{sd_}: da co, bo qua'); continue
        t0 = time.time(); pv, ps = fit(X, seed=sd_)
        a = ap_cols(YM[VAk], pv)
        row = dict(arm=tag, mo_ta=desc, seed=sd_, D=X.shape[1],
                   mAP_anh=float(np.nanmean(a)), **tiers(a),
                   sel=float(np.nanmean(ap_cols(YM[SLk], ps))))
        R = pd.concat([R, pd.DataFrame([row])], ignore_index=True)
        ensure_res(); R.to_csv(P_RES, index=False)
        np.save(os.path.join(RES44A, f'p_{tag}_s{sd_}.npy'), pv)
        print(f'{tag} s{sd_}  D={X.shape[1]:>4}  mAP {row["mAP_anh"]:.4f}  tail {row["tail"]:.4f}  '
              f'lat chon {row["sel"]:.4f}  ({time.time()-t0:.0f}s)')
    del X
print('\nxong')

A s42  D= 768  mAP 0.2183  tail 0.0641  lat chon 0.2145  (3s)
A s43  D= 768  mAP 0.2176  tail 0.0622  lat chon 0.2139  (3s)
A s44  D= 768  mAP 0.2182  tail 0.0642  lat chon 0.2139  (3s)
B s42  D=1536  mAP 0.2179  tail 0.0558  lat chon 0.2177  (3s)
B s43  D=1536  mAP 0.2173  tail 0.0539  lat chon 0.2162  (3s)
B s44  D=1536  mAP 0.2186  tail 0.0595  lat chon 0.2173  (3s)
D s42  D=1536  mAP 0.2189  tail 0.0572  lat chon 0.2182  (3s)
D s43  D=1536  mAP 0.2188  tail 0.0571  lat chon 0.2164  (3s)
D s44  D=1536  mAP 0.2194  tail 0.0587  lat chon 0.2173  (3s)
Z s42  D=1536  mAP 0.2100  tail 0.0530  lat chon 0.2067  (3s)
Z s43  D=1536  mAP 0.2097  tail 0.0538  lat chon 0.2057  (3s)
Z s44  D=1536  mAP 0.2101  tail 0.0539  lat chon 0.2066  (3s)

xong


## C5 · Đọc kết quả — hai câu hỏi, và **biên độ seed** để `44b` mượn

In [6]:
R = pd.read_csv(P_RES)
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel'], aggfunc='mean')
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min())
piv['D'] = R.groupby('arm').D.first()
print('=== TUYET DOI (TB qua 3 seed) ===')
print(piv.reindex(list(ARMS)).to_string(float_format=lambda v: f'{v:.4f}'))
NOISE = float(piv.bien_do.max())
print(f'\nBIEN DO SEED lon nhat qua cac arm: {NOISE:.4f}')
print('  <- day la con so `44b` se MUON. Ghi ro trong REPORT la bien do MUON, khong do tai cho.')

g = lambda t, s: float(R[(R.arm==t)&(R.seed==s)].mAP_anh.iloc[0])
print('\n=== HIEU GHEP CAP ===')
for a, b, ngh in (('B','A','anh truoc THO (co the chi la them CHIEU)'),
                  ('D','A','HIEU (dong cung phep tru)'),
                  ('Z','A','DOI CHUNG: them chieu, phim truoc SAI benh nhan'),
                  ('B','Z','loi ich cua THO co den tu DUNG benh nhan khong'),
                  ('D','B','<<< CAU HOI 1: HIEU co hon THO tren backbone `ft` khong')):
    d = [g(a,s)-g(b,s) for s in SEEDS]
    ok = all(x>0 for x in d) or all(x<0 for x in d)
    dat = 'DAT' if (ok and abs(np.mean(d)) > (max(d)-min(d))) else '---'
    print(f'  {a} - {b}  {np.mean(d):+.4f}  (bien do {max(d)-min(d):.4f}, '
          f'{"cung dau" if ok else "DOI DAU"}, {dat})   {ngh}')

print('\n=== DEM SO NHAN ===')
_pa = np.mean([np.load(os.path.join(RES44A, f'p_A_s{s}.npy')) for s in SEEDS], 0)
_aa = ap_cols(YM[VAk], _pa)
for t in ('B','D','Z'):
    _pt = np.mean([np.load(os.path.join(RES44A, f'p_{t}_s{s}.npy')) for s in SEEDS], 0)
    dl = ap_cols(YM[VAk], _pt) - _aa
    k = int((dl > NOISE).sum())
    print(f'  {t} - A: {k:>2}/40 nhan vuot bien do | trung vi {np.nanmedian(dl):+.5f} '
          f'| macro can neu k={k}: {0.0007*40/max(k,1):.4f}')
    if t == 'D':
        print('     nhan len nhieu nhat: ' +
              ' · '.join(f'{LAB40[j]} {dl[j]:+.4f}' for j in np.argsort(-dl)[:6]))
print('\n  43 (backbone PRETRAIN, mean-pool) da do: D +0.0031 > B +0.0026 > C +0.0010, Z -0.0089')
print('  D > B lap lai  -> `44b` dong cung phep hieu vao Tier1Img la CO CO SO')
print('  D ~ B hoac nguoc lai -> `44b` phai thiet ke lai, va biet dieu do sau 40 phut chu khong 3.5 gio')

=== TUYET DOI (TB qua 3 seed) ===
      head  mAP_anh  medium    sel   tail  bien_do     D
arm                                                     
A   0.4408   0.2180  0.1433 0.2141 0.0635   0.0007   768
B   0.4444   0.2179  0.1449 0.2171 0.0564   0.0013  1536
D   0.4452   0.2191  0.1463 0.2173 0.0577   0.0006  1536
Z   0.4366   0.2099  0.1333 0.2064 0.0535   0.0004  1536

BIEN DO SEED lon nhat qua cac arm: 0.0013
  <- day la con so `44b` se MUON. Ghi ro trong REPORT la bien do MUON, khong do tai cho.

=== HIEU GHEP CAP ===
  B - A  -0.0001  (bien do 0.0007, DOI DAU, ---)   anh truoc THO (co the chi la them CHIEU)
  D - A  +0.0010  (bien do 0.0006, cung dau, DAT)   HIEU (dong cung phep tru)
  Z - A  -0.0081  (bien do 0.0005, cung dau, DAT)   DOI CHUNG: them chieu, phim truoc SAI benh nhan
  B - Z  +0.0080  (bien do 0.0009, cung dau, DAT)   loi ich cua THO co den tu DUNG benh nhan khong
  D - B  +0.0011  (bien do 0.0007, cung dau, DAT)   <<< CAU HOI 1: HIEU co hon THO tren backbone `ft

## C6 · Nhóm nhãn nhiễm trùng + ghi `REVIEW_PRIOR_MEANPOOL.md`

In [7]:
INF = [l for l in LAB40 if re.search(r'pneumon|tubercul|infect|consolidat|abscess|empyema', l, re.I)]
IDX_INF = np.array([LAB40.index(l) for l in INF], dtype=int)
print(f'nhom nhiem trung ({len(INF)}): {INF}')
T6 = []
for t in ARMS:
    _p = np.mean([np.load(os.path.join(RES44A, f'p_{t}_s{s}.npy')) for s in SEEDS], 0)
    a = ap_cols(YM[VAk], _p)
    T6.append(dict(arm=t, macro=float(np.nanmean(a)),
                   nhiem_trung=float(np.nanmean(a[IDX_INF])) if len(IDX_INF) else np.nan,
                   tail=float(np.nanmean(a[TIER['tail']]))))
T6 = pd.DataFrame(T6).set_index('arm')
for c in ('macro','nhiem_trung','tail'): T6[f'd_{c}'] = T6[c] - T6.loc['A', c]
print(T6.to_string(float_format=lambda v: f'{v:+.4f}'))
print('\n  DOC: `Z` cung tap trung o nhiem trung thi phan tap trung do KHONG den tu phim truoc.')
print('       (38d da bat dung loi nay: t1_zero khong co EHR ma van tap trung 3.8x.)')

L = []; w = L.append
w('# REVIEW · phim chụp trước ở mức `mean(patch)`, backbone **ft** (`nb44a`)'); w('')
w(f'`38a/{BB_RUN}` · `asl40 E={EPOCHS}` epoch cuối · seeds {list(SEEDS)} · '
  f'chỉ ảnh **có** phim trước (val {int(VAk.sum()):,}).'); w('')
w('| arm | biểu diễn | D | mAP ảnh | biên độ | tail | nhiễm trùng |')
w('|---|---|---|---|---|---|---|')
for t in ARMS:
    r = R[R.arm == t]
    w(f'| `{t}` | `{ARMS[t]}` | {int(r.D.iloc[0])} | {r.mAP_anh.mean():.4f} | '
      f'{r.mAP_anh.max()-r.mAP_anh.min():.4f} | {r["tail"].mean():.4f} | '
      f'{T6.loc[t,"nhiem_trung"]:.4f} |')
w(''); w('```')
for a, b in (('D','B'), ('D','A'), ('B','A'), ('Z','A'), ('B','Z')):
    d = [g(a,s)-g(b,s) for s in SEEDS]
    w(f'{a} - {b}  {np.mean(d):+.4f}  bien do {max(d)-min(d):.4f}')
w(f'BIEN DO SEED lon nhat: {NOISE:.4f}   <- `44b` MUON con so nay'); w('```'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **Mức patch khác hẳn.** `mean(patch)` xoá sạch toạ độ nên phép trừ hợp lệ. Ở `44b` (1369 token) '
  'hai phim **không được đăng ký**, nên phép trừ theo từng ô là nhiễu lệch vị trí — phải dùng '
  'cross-attention. Kết luận `D > B` ở đây **không** tự động chuyển sang đó, nó chỉ nói việc đóng '
  'cứng phép hiệu là **đáng thử**.')
w('- **Cơ chế.** `42` đo `p25 = 0.9 ngày` — một phần tư phim trước là phim nối tiếp trong cùng đợt '
  'nằm viện, nơi nhãn gần như bất biến. Lợi ích (nếu có) phần lớn là **tự tương quan nhãn trong một '
  'đợt điều trị**, không phải phát hiện thay đổi.')
w(f'- **{100-100*float(HAS.mean()):.1f}% ảnh không có phim trước** bị loại khỏi phép đo này.')
w('- Một backbone, một phép gộp, một loss.')
TXT = '\n'.join(L)
ensure_res(); open(os.path.join(RES44A, 'REVIEW_PRIOR_MEANPOOL.md'), 'w', encoding='utf-8').write(TXT)
print('\n' + TXT)

nhom nhiem trung (3): ['Consolidation', 'Pneumonia', 'Tuberculosis']
      macro  nhiem_trung    tail  d_macro  d_nhiem_trung  d_tail
arm                                                             
A   +0.2186      +0.1744 +0.0641  +0.0000        +0.0000 +0.0000
B   +0.2185      +0.1732 +0.0573  -0.0001        -0.0012 -0.0068
D   +0.2197      +0.1748 +0.0586  +0.0011        +0.0004 -0.0056
Z   +0.2105      +0.1714 +0.0538  -0.0081        -0.0030 -0.0103

  DOC: `Z` cung tap trung o nhiem trung thi phan tap trung do KHONG den tu phim truoc.
       (38d da bat dung loi nay: t1_zero khong co EHR ma van tap trung 3.8x.)

# REVIEW · phim chụp trước ở mức `mean(patch)`, backbone **ft** (`nb44a`)

`38a/a2_E3` · `asl40 E=60` epoch cuối · seeds [42, 43, 44] · chỉ ảnh **có** phim trước (val 26,397).

| arm | biểu diễn | D | mAP ảnh | biên độ | tail | nhiễm trùng |
|---|---|---|---|---|---|---|
| `A` | `[e_cur]` | 768 | 0.2180 | 0.0007 | 0.0635 | 0.1744 |
| `B` | `[e_cur;e_prior]` | 1536 | 0.217